Dec-POMDP Formulation & CTDE Architecture for MAPPO

In single-agent RL (PPO, SAC), the environment is assumed to be stationary: given state $s$ and action $a$, the probability distribution over the next state $s'$ is fixed.

# 1. Start with the actual problem

Imagine two robots need to push a box through a door.

```text
        DOOR
         ↓
     ┌────────┐
     │  BOX   │
     └────────┘

   🤖 A       🤖 B
```

Both robots need to cooperate.

Now suppose A can only see:

```text
A's camera:
    BOX
     ↑
    🤖A
```

And B can only see its own surroundings.

Your question is:

> **If A doesn't know what B is doing, HOW THE HELL can A coordinate with B?**

Exactly.

## Answer: it doesn't necessarily need to know B's full state.

There are several ways coordination can happen.

---

# 2. Simplest case: the environment itself gives enough information

Suppose the rules are:

> **If A pushes left, B should push right.**

A doesn't need to know B's exact position.

It can learn:

```text
A sees:
box is on my right
        ↓
A → push right
```

B independently sees:

```text
box is on my left
        ↓
B → push left
```

They coordinate because their **local observations + learned policies + shared objective** produce compatible actions.

Think of two people carrying a table.

Person A doesn't necessarily need a live dashboard saying:

```text
B's exact hand position = (1.72, 3.41)
B's velocity = 0.83 m/s
B's intended action = ...
```

A can simply see enough of the table/person/environment to act appropriately.

---

# 3. But what if A REALLY needs to know B?

Now we have an important distinction.

There are two possibilities.

### Case A — A can observe B

Then B's position might simply be part of A's observation:

$$
o_A =
[\text{A position},\text{B position},\text{box position}]
$$

No problem.

A **does know about B**.

---

### Case B — A cannot observe B

Then:

$$
o_A =
[\text{A position},\text{box}]
$$

A genuinely doesn't know where B is.

Now coordination is harder.

But it can still learn coordination through **interaction history**.

For example:

```text
t=0
A does X
B does Y
→ reward +10

t=1
A does X
B does Z
→ reward -10
```

Over many episodes, A can learn:

> "When I see this situation, doing X tends to work."

It doesn't necessarily learn:

> "B is currently at coordinate (4,7)."

It learns a policy based on what **it can actually observe**.

---

# 4. Here's where your intuition is 100% correct

If B's hidden state is **essential** for A's decision, then A has a problem.

Imagine:

```text
A sees:

       BOX
        |
        A
```

But B could secretly be either:

```text
Situation 1:

B → BOX ← A


Situation 2:

B
↓
BOX ← A
```

A sees exactly the same thing.

But the correct action is different depending on B.

A cannot reliably choose the correct action.

Why?

Because **the information needed to make the decision isn't in \(o_A\)**.

No algorithm can magically recover information that isn't observable.

This is one of the fundamental problems of **partial observability**.

---

# 5. So how does MAPPO help?

Here's the clever part.

During **training**, we can give the critic information that the actors don't have.

Suppose the actual situation is:

```text
                 GLOBAL WORLD

        B
        ↓
       BOX ← A
```

The actors receive:

```text
A:
o_A = what A can see

B:
o_B = what B can see
```

But the critic receives:

```text
s = entire world
```

So:

```text
               GLOBAL STATE
                    │
                    ▼
             Central Critic
                  V(s)
                    │
                    │
        ┌───────────┴───────────┐
        ▼                       ▼
      Actor A                 Actor B
       o_A                     o_B
        ↓                       ↓
       a_A                     a_B
```

The critic knows:

> "Ah, B was actually behind the box."

A doesn't.

---

# 6. But wait — doesn't that STILL mean A can't coordinate?

**Yes, potentially.**

And this is extremely important:

> **CTDE does NOT magically solve partial observability.**

The centralized critic helps the **learning process**.

It does not give A hidden information at execution time.

If A genuinely needs B's hidden state to act correctly, you need something additional, such as:

* communication between agents
* better observations
* recurrent policies / memory
* belief states
* explicit communication actions

MAPPO itself isn't telepathy. 😭

---

# 7. Then why is the centralized critic useful?

Because training a partially observed multi-agent system is noisy as hell.

Consider A:

```text
A takes action LEFT
```

Then gets:

```text
reward = -10
```

Why?

Was A's action bad?

Or did B screw up?

Or was the environment already in a bad state?

A local critic only sees:

```text
o_A → V(o_A)
```

It doesn't have the whole picture.

Centralized critic sees:

```text
s
+
joint situation
→
V(s)
```

So it can produce a **better estimate of the team's situation**.

That gives A a better advantage estimate during training.

---

# 8. The key distinction

This is the thing I want you to lock in:

### Actor

Answers:

> **"Given what I can currently see, what should I do?"**

$$
a_i \sim \pi_i(a_i|o_i)
$$

### Critic

Answers:

> **"Given the whole situation, how good is this situation?"**

$$
V(s)
$$

So:

```text
EXECUTION:

A only gets → o_A → Actor A → a_A
B only gets → o_B → Actor B → a_B


TRAINING:

A's observation ─────→ Actor A
B's observation ─────→ Actor B
                           │
Global state ─────────→ Critic
                           │
                           ↓
                    better advantage
```

---

# 9. And now coordination makes more sense

Coordination doesn't necessarily mean:

> "A must know exactly what B is doing."

It can mean:

> "A and B have learned policies whose actions work well together."

For example, imagine traffic.

Two drivers don't need to know each other's neural-network internals.

They coordinate because they have:

* observations
* shared rules
* environment signals
* learned behavior
* possibly communication

Multi-agent RL is basically trying to learn this kind of **joint behavior**.

---

# 10. One final example

Imagine a game:

```text
A                B

🚗 →      🏁      ← 🚗
```

Reward:

```text
+100 if both reach the goal
-100 if they collide
```

A doesn't see B.

During training:

```text
A observation → Actor A → action A
B observation → Actor B → action B

                ↓
             Environment
                ↓
             reward
                ↓
       Global state → Critic
```

After millions of interactions, A may learn:

> "When I see this configuration, moving right tends to produce good team outcomes."

B independently learns its compatible behavior.

**That's coordination without direct knowledge of the other's internal state.**

But if the problem fundamentally requires hidden information from B, **we need communication or memory**. That's not a MAPPO failure; it's an information constraint.

---

if every agent gets the full global state, it's basically multi-agent PPO with centralized observations.

Classic MAPPO is specifically:

$$ \boxed{\text{Decentralized Actors} + \text{Centralized Critic}} $$

If actors also get the full state:

$$ \boxed{\text{Centralized Actors} + \text{Centralized Critic}} $$

So the PPO optimization machinery is still there, but you've removed the key decentralized-execution constraint that makes MAPPO interesting.

PettingZoo Multi-Agent Environment Setup